In [65]:
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd
con = duckdb.connect("warehouse.duckdb")

In [66]:
con.execute("""
    DROP TABLE IF EXISTS fact_sales;
    DROP TABLE IF EXISTS dim_time;
    DROP TABLE IF EXISTS dim_item;
    DROP TABLE IF EXISTS dim_location;
            
    -- DIMENSIONES — atributos descriptivos y jerarquías
    CREATE TABLE dim_time (
    time_key INTEGER PRIMARY KEY,
    day DATE, month VARCHAR, quarter VARCHAR, year INTEGER );
            
    CREATE TABLE dim_item (
    item_key INTEGER PRIMARY KEY,
    item_name VARCHAR, brand VARCHAR, category VARCHAR );
            
    CREATE TABLE dim_location (
    loc_key INTEGER PRIMARY KEY,
    city VARCHAR, state VARCHAR, country VARCHAR );
    -- HECHOS — claves foraneas + medidas numericas
            
    CREATE TABLE fact_sales (
    time_key INTEGER REFERENCES dim_time(time_key),
    item_key INTEGER REFERENCES dim_item(item_key),
    loc_key INTEGER REFERENCES dim_location(loc_key),
    dollars_sold DECIMAL(12,2),
    units_sold INTEGER );
""")

In [67]:
con.execute("""
    INSERT INTO dim_time SELECT * FROM read_csv_auto('data/dim_time.csv');
    INSERT INTO dim_item SELECT * FROM read_csv_auto('data/dim_item.csv');
    INSERT INTO dim_location SELECT * FROM read_csv_auto('data/dim_location.csv');
    INSERT INTO fact_sales SELECT * FROM read_csv_auto('data/fact_sales.csv');
""")

In [68]:
CSV_PATH = {
    "dim_item": "./data/dim_item.csv",
    "dim_location": "./data/dim_location.csv",
    "dim_time": "./data/dim_time.csv",
    "fact_sales": "./data/fact_sales.csv"
}
if not Path(CSV_PATH["dim_item"]).exists() or not Path(CSV_PATH["dim_location"]).exists() or not Path(CSV_PATH["dim_time"]).exists() or not Path(CSV_PATH["fact_sales"]).exists():
    CSV_PATH = {
        "dim_item": "../data/dim_item.csv",
        "dim_location": "../data/dim_location.csv",
        "dim_time": "../data/dim_time.csv",
        "fact_sales": "../data/fact_sales.csv"
    }
keys = ['dim_item', 'dim_location', 'dim_time', 'fact_sales']

for i, key in enumerate(keys, start=1):
    print(f"Tabla {i}: {key}")
    df = pd.read_csv(CSV_PATH[key])
    print(f"Shape: {df.shape}")
    print(df.head(4), "\n")


Tabla 1: dim_item
Shape: (50, 4)
   item_id            item_name     brand     category
0        1  Electronics Item 01  NovaTech  Electronics
1        2  Electronics Item 02  PixelPro  Electronics
2        3  Electronics Item 03  NovaTech  Electronics
3        4  Electronics Item 04  PixelPro  Electronics 

Tabla 2: dim_location
Shape: (30, 4)
   location_id              city    state country
0            1  Tuxtla Gutierrez  Chiapas  Mexico
1            2     San Cristobal  Chiapas  Mexico
2            3         Tapachula  Chiapas  Mexico
3            4            Merida  Yucatan  Mexico 

Tabla 3: dim_time
Shape: (365, 5)
   time_id         day  month  quarter  year
0        1  2025-01-01      1        1  2025
1        2  2025-01-02      1        1  2025
2        3  2025-01-03      1        1  2025
3        4  2025-01-04      1        1  2025 

Tabla 4: fact_sales
Shape: (10000, 5)
   time_id  item_id  location_id  dollars_sold  units_sold
0      159       24           17        133